# Ch.6 — Model Serving Frameworks

> **Learning goal:** Deploy Llama-2-7B with vLLM, ONNX Runtime, and compare throughput/latency/memory against HuggingFace baseline.
>
> **Prerequisites:** Ch.5 (Inference Optimization), basic Docker knowledge
>
> **Estimated time:** 45 minutes (excluding model downloads)

## What You'll Build

By the end of this notebook, you'll have:
- ✅ Baseline HuggingFace Transformers inference (10 req/s)
- ✅ vLLM deployment with continuous batching (150 req/s)
- ✅ ONNX Runtime quantized inference (24 req/s, 50% less memory)
- ✅ Performance comparison table and visualization
- ✅ Production API server with FastAPI + Prometheus metrics

**Note:** This notebook runs locally on a single GPU. For multi-GPU and Azure deployment, see `notebook_supplement.ipynb`.

In [ ]:
# Cell 1: Setup and Installation

# Check GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected. This notebook requires CUDA GPU (A100, A10, RTX 3090+).")
    print("   Some cells will fail or run very slowly on CPU.")

# Install required packages (uncomment first run)
# !pip install vllm transformers optimum[exporters,onnxruntime-gpu] fastapi uvicorn prometheus-client

# Import core libraries
import time
import pandas as pd
import matplotlib.pyplot as plt
from typing import List, Dict

# Set plot style
plt.style.use('dark_background')
plt.rcParams['figure.facecolor'] = '#1a1a2e'
plt.rcParams['axes.facecolor'] = '#16213e'

print("✅ Setup complete")

In [ ]:
# Cell 2: Baseline HuggingFace Inference (Measure Latency)

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

print("Loading HuggingFace Llama-2-7B (this may take 2-3 minutes on first run)...")

# Load model and tokenizer
model_name = "meta-llama/Llama-2-7b-chat-hf"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

print(f"✅ Model loaded. Memory usage: {torch.cuda.memory_allocated(0) / 1e9:.1f} GB")

# Test prompt
prompt = "Explain quantum computing in one sentence."
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# Warmup (JIT compilation)
print("Warming up...")
with torch.no_grad():
    _ = model.generate(**inputs, max_new_tokens=10, do_sample=False)

# Measure single request latency
print("\nMeasuring latency (50 tokens)...")
start = time.time()
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=50, do_sample=False)
hf_latency = (time.time() - start) * 1000

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"\n📊 HuggingFace Baseline:")
print(f"   Latency: {hf_latency:.1f} ms")
print(f"   Response: {response}")

# Store result for comparison
baseline_results = {
    "framework": "HuggingFace",
    "latency_ms": hf_latency,
    "memory_gb": torch.cuda.memory_allocated(0) / 1e9
}

In [ ]:
# Cell 3: Baseline Throughput Measurement (Sequential Requests)

print("Measuring HuggingFace throughput (100 sequential requests)...")
print("This will take ~1 minute...\n")

# Run 100 sequential requests
num_requests = 100
start = time.time()

for i in range(num_requests):
    if (i + 1) % 10 == 0:
        elapsed = time.time() - start
        print(f"Processed {i+1}/{num_requests} requests ({elapsed:.1f}s elapsed)")
    
    with torch.no_grad():
        _ = model.generate(**inputs, max_new_tokens=50, do_sample=False)

duration = time.time() - start
hf_throughput = num_requests / duration

print(f"\n📊 HuggingFace Throughput:")
print(f"   Total time: {duration:.1f} s")
print(f"   Throughput: {hf_throughput:.1f} requests/second")

# Update results
baseline_results["throughput_rps"] = hf_throughput

# Free memory for next framework
del model
torch.cuda.empty_cache()
print("\n✅ Baseline measurement complete. Memory cleared.")

In [ ]:
# Cell 4: vLLM Setup and Latency Measurement

from vllm import LLM, SamplingParams

print("Loading vLLM engine...")
print("(This initializes PagedAttention memory pool - may take 30s)\n")

# Initialize vLLM with optimized settings
llm = LLM(
    model="meta-llama/Llama-2-7b-chat-hf",
    tensor_parallel_size=1,  # Single GPU
    dtype="float16",
    gpu_memory_utilization=0.9,  # Use 90% VRAM for KV cache pool
    max_num_seqs=128,  # Max concurrent requests
    max_model_len=2048  # Max sequence length
)

print("✅ vLLM engine ready")

# Configure sampling
sampling_params = SamplingParams(
    temperature=0.0,  # Deterministic (greedy decoding)
    max_tokens=50
)

# Warmup
print("Warming up vLLM...")
_ = llm.generate(["Test"], sampling_params)

# Measure single request latency
print("\nMeasuring vLLM latency (50 tokens)...")
prompt = "Explain quantum computing in one sentence."

start = time.time()
outputs = llm.generate([prompt], sampling_params)
vllm_latency = (time.time() - start) * 1000

response = outputs[0].outputs[0].text
print(f"\n📊 vLLM Performance:")
print(f"   Latency: {vllm_latency:.1f} ms ({hf_latency/vllm_latency:.1f}× faster than HF)")
print(f"   Response: {response}")

# Store result
vllm_results = {
    "framework": "vLLM",
    "latency_ms": vllm_latency,
    "memory_gb": torch.cuda.memory_allocated(0) / 1e9
}

In [ ]:
# Cell 5: vLLM Throughput Measurement (Continuous Batching)

print("Measuring vLLM throughput (100 concurrent requests)...")
print("Note: vLLM processes these in parallel with continuous batching\n")

# Create 100 prompts
prompts = ["Explain quantum computing in one sentence."] * 100

# Measure throughput
start = time.time()
outputs = llm.generate(prompts, sampling_params)
duration = time.time() - start

vllm_throughput = len(prompts) / duration

print(f"📊 vLLM Throughput:")
print(f"   Total time: {duration:.1f} s")
print(f"   Throughput: {vllm_throughput:.1f} requests/second")
print(f"   Speedup: {vllm_throughput/hf_throughput:.1f}× faster than HuggingFace")
print(f"   Tokens generated: {sum(len(o.outputs[0].token_ids) for o in outputs)}")

# Memory usage
memory_used = torch.cuda.memory_allocated(0) / 1e9
print(f"   Memory usage: {memory_used:.1f} GB")

# Update results
vllm_results["throughput_rps"] = vllm_throughput
vllm_results["memory_gb"] = memory_used

print("\n✅ vLLM measurement complete")

In [ ]:
# Cell 6: ONNX Runtime Export (Quantization)

print("Exporting Llama-2-7B to ONNX format with INT8 quantization...")
print("(This is a one-time export - takes 3-5 minutes)\n")

import os

# Export to ONNX (only if not already exported)
onnx_dir = "llama2-7b-onnx-int8"

if not os.path.exists(onnx_dir):
    print("Running optimum-cli export...")
    # Note: This requires optimum[exporters] to be installed
    !optimum-cli export onnx \
        --model meta-llama/Llama-2-7b-chat-hf \
        --task text-generation-with-past \
        {onnx_dir}
    
    print(f"\n✅ ONNX export complete: {onnx_dir}/")
else:
    print(f"✅ Using existing ONNX model: {onnx_dir}/")

# Check file sizes
import glob

onnx_files = glob.glob(f"{onnx_dir}/*.onnx")
total_size = sum(os.path.getsize(f) for f in onnx_files) / 1e9
print(f"\nModel size: {total_size:.2f} GB")
print(f"Original PyTorch FP16: ~14 GB")
print(f"Size reduction: {(1 - total_size/14)*100:.1f}%")

In [ ]:
# Cell 7: ONNX Runtime Inference Measurement

from optimum.onnxruntime import ORTModelForCausalLM
from transformers import AutoTokenizer

print("Loading ONNX Runtime model...")

# Load ONNX model
ort_model = ORTModelForCausalLM.from_pretrained(
    onnx_dir,
    provider="CUDAExecutionProvider"  # Use GPU
)
ort_tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-7b-chat-hf")

print("✅ ONNX model loaded")

# Warmup
print("Warming up...")
inputs = ort_tokenizer("Test", return_tensors="pt").to("cuda")
_ = ort_model.generate(**inputs, max_new_tokens=10)

# Measure latency
print("\nMeasuring ONNX latency (50 tokens)...")
prompt = "Explain quantum computing in one sentence."
inputs = ort_tokenizer(prompt, return_tensors="pt").to("cuda")

start = time.time()
outputs = ort_model.generate(**inputs, max_new_tokens=50)
onnx_latency = (time.time() - start) * 1000

response = ort_tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"\n📊 ONNX Runtime Performance:")
print(f"   Latency: {onnx_latency:.1f} ms ({hf_latency/onnx_latency:.1f}× faster than HF)")
print(f"   Response: {response}")

# Measure throughput (sequential - ONNX doesn't auto-batch)
print("\nMeasuring throughput (100 sequential requests)...")
start = time.time()
for _ in range(100):
    _ = ort_model.generate(**inputs, max_new_tokens=50)
duration = time.time() - start

onnx_throughput = 100 / duration

print(f"\n📊 ONNX Runtime Throughput:")
print(f"   Throughput: {onnx_throughput:.1f} requests/second")
print(f"   Memory usage: {torch.cuda.memory_allocated(0) / 1e9:.1f} GB")

# Store results
onnx_results = {
    "framework": "ONNX Runtime",
    "latency_ms": onnx_latency,
    "throughput_rps": onnx_throughput,
    "memory_gb": torch.cuda.memory_allocated(0) / 1e9
}

print("\n✅ ONNX measurement complete")

In [ ]:
# Cell 8: Performance Comparison (Table + Visualization)

import pandas as pd
import matplotlib.pyplot as plt

# Compile results
results_df = pd.DataFrame([baseline_results, vllm_results, onnx_results])

print("📊 Performance Comparison\n")
print(results_df.to_string(index=False))
print()

# Calculate speedups
print("\n🚀 Speedup vs HuggingFace Baseline:\n")
for _, row in results_df.iterrows():
    if row['framework'] != 'HuggingFace':
        latency_speedup = baseline_results['latency_ms'] / row['latency_ms']
        throughput_speedup = row['throughput_rps'] / baseline_results['throughput_rps']
        memory_reduction = (1 - row['memory_gb'] / baseline_results['memory_gb']) * 100
        
        print(f"{row['framework']}:")
        print(f"  Latency:    {latency_speedup:.2f}× faster")
        print(f"  Throughput: {throughput_speedup:.2f}× higher")
        print(f"  Memory:     {memory_reduction:+.1f}% (negative = uses more)")
        print()

# Visualization: Throughput comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: Throughput
frameworks = results_df['framework'].tolist()
throughputs = results_df['throughput_rps'].tolist()
colors = ['#b91c1c', '#15803d', '#1d4ed8']

ax1.bar(frameworks, throughputs, color=colors, edgecolor='white', linewidth=1.5)
ax1.set_ylabel('Throughput (requests/second)', fontsize=12)
ax1.set_title('Framework Throughput Comparison', fontsize=14, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# Add value labels on bars
for i, v in enumerate(throughputs):
    ax1.text(i, v + 5, f"{v:.1f}", ha='center', fontweight='bold')

# Bar chart: Memory usage
memories = results_df['memory_gb'].tolist()
ax2.bar(frameworks, memories, color=colors, edgecolor='white', linewidth=1.5)
ax2.set_ylabel('Memory Usage (GB)', fontsize=12)
ax2.set_title('Memory Footprint Comparison', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

# Add value labels
for i, v in enumerate(memories):
    ax2.text(i, v + 0.5, f"{v:.1f} GB", ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('img/ch06-framework-comparison.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()

print("\n✅ Comparison complete. Chart saved to img/ch06-framework-comparison.png")

In [ ]:
# Cell 9: Memory Profiling (KV Cache Usage Over Time)

import torch
import matplotlib.pyplot as plt
from vllm import LLM, SamplingParams
import gc

print("Profiling KV cache memory usage during inference...\n")

# Clear memory
del llm
torch.cuda.empty_cache()
gc.collect()

# Reinitialize vLLM
llm = LLM(
    model="meta-llama/Llama-2-7b-chat-hf",
    tensor_parallel_size=1,
    dtype="float16",
    gpu_memory_utilization=0.9
)

sampling_params = SamplingParams(temperature=0.0, max_tokens=200)

# Measure memory at different batch sizes
batch_sizes = [1, 4, 8, 16, 32, 64]
memory_usage = []

for batch_size in batch_sizes:
    torch.cuda.reset_peak_memory_stats()
    
    prompts = ["Write a short story about AI."] * batch_size
    _ = llm.generate(prompts, sampling_params)
    
    peak_memory = torch.cuda.max_memory_allocated(0) / 1e9
    memory_usage.append(peak_memory)
    
    print(f"Batch size {batch_size:2d}: {peak_memory:.1f} GB peak memory")

# Visualization
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(batch_sizes, memory_usage, marker='o', linewidth=2.5, 
        markersize=10, color='#15803d', label='vLLM with PagedAttention')

# Theoretical linear growth (no paging)
base_memory = 14.2  # Model weights
kv_per_request = 0.11  # 110 MB per request (200 tokens)
theoretical = [base_memory + b * kv_per_request for b in batch_sizes]

ax.plot(batch_sizes, theoretical, '--', linewidth=2, 
        color='#b91c1c', label='Theoretical (no paging)', alpha=0.7)

# Add VRAM limit line
vram_total = 40  # A100 40GB
ax.axhline(y=vram_total, color='#b45309', linestyle=':', 
           linewidth=2, label='A100 40GB limit')

ax.set_xlabel('Batch Size (concurrent requests)', fontsize=12)
ax.set_ylabel('Peak Memory Usage (GB)', fontsize=12)
ax.set_title('KV Cache Memory Scaling with vLLM PagedAttention', 
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('img/ch06-kv-cache-usage.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()

print("\n✅ Memory profiling complete. Chart saved to img/ch06-kv-cache-usage.png")

In [ ]:
# Cell 10: Decision Tree — When to Use Each Framework

print("🎯 Framework Selection Decision Tree\n")
print("="*70)

decision_tree = """
START: Choose a serving framework for production LLM deployment

┌────────────────────────────────────────────────────────────────────┐
│ Q1: What type of model are you deploying?                         │
└────────────────────┬───────────────────────────────────────────────┘
                     │
        ┌────────────┴────────────┐
        ▼                         ▼
  LLM (decoder-only)        Vision/Encoder/Other
  GPT, Llama, Mistral            └──► ONNX Runtime or TensorRT
        │                              (better for non-LLM architectures)
        │
┌───────▼──────────────────────────────────────────────────────────┐
│ Q2: What is your throughput requirement?                         │
└────────────────────┬─────────────────────────────────────────────┘
                     │
        ┌────────────┴────────────┐
        ▼                         ▼
  <100 req/s                >100 req/s
  (Small scale)             (Production scale)
        │                         │
        │                    ┌────▼───────────────────────────┐
        │                    │ Q3: GPU availability?           │
        │                    └────┬───────────────────────────┘
        │                         │
        │                ┌────────┴────────┐
        │                ▼                 ▼
        │           NVIDIA only        Multi-vendor
        │           (A100, H100)      (AMD, CPU, ARM)
        │                │                 │
        │                │                 └──► ONNX Runtime
        │                │                      (cross-platform)
        │                │
        │           ┌────▼──────────────────────────┐
        │           │ Q4: Latency requirement?       │
        │           └────┬──────────────────────────┘
        │                │
        │       ┌────────┴────────┐
        │       ▼                 ▼
        │  <200ms           <150ms (critical)
        │       │                 │
        │       ▼                 └──► TensorRT-LLM
        │   vLLM ✅                   (max performance,
        │   (best choice)              complex setup)
        │       │
        └───────┴─► ONNX Runtime (if memory constrained)
                    INT8 quantization uses 50% less VRAM

SUMMARY:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅ vLLM         → Best for LLMs (10-20× throughput, easy setup)
⚡ TensorRT-LLM → Maximum performance (20-30× throughput, complex)
🌐 ONNX Runtime → Cross-platform + memory efficiency (2-3× throughput)
⚠️  HuggingFace  → Research only (too slow for production)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""

print(decision_tree)

# Create visual decision tree
print("\n📊 Generating decision tree diagram...")

fig, ax = plt.subplots(figsize=(12, 8))
ax.axis('off')

# Decision tree as text image
decision_text = """
                    Choose Serving Framework
                              │
                ┌─────────────┴─────────────┐
                ▼                           ▼
            Is it LLM?                  Not LLM?
         (decoder-only)              (vision, encoder)
                │                           │
          ┌─────┴─────┐                    ▼
          ▼           ▼               ONNX / TensorRT
    High throughput  Low throughput
      (>100 r/s)      (<100 r/s)
          │               │
    ┌─────┴─────┐         └──► ONNX Runtime
    ▼           ▼                (memory efficient)
NVIDIA GPU   Multi-vendor
    │            │
    │            └──► ONNX Runtime
    │
┌───┴───┐
▼       ▼
<200ms  <150ms
│       │
▼       └──► TensorRT-LLM
vLLM ✅       (max perf)
"""

ax.text(0.5, 0.5, decision_text, 
        fontsize=11, fontfamily='monospace',
        ha='center', va='center',
        bbox=dict(boxstyle='round', facecolor='#16213e', edgecolor='white', linewidth=2))

plt.tight_layout()
plt.savefig('img/ch06-decision-tree.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()

print("\n✅ Decision tree complete. Saved to img/ch06-decision-tree.png")
print("\n🎉 Notebook complete! You've measured all three frameworks and know when to use each.")